# Seq2Seq English to Hindi Translation
This notebook trains a sequence-to-sequence model on a large dataset and saves the weights for external inference.


## 1. Import Libraries & GPU Setup
Load TensorFlow and configure GPU memory growth to prevent Out-of-Memory crashes over time.


In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import json
import collections
import tensorflow.keras.preprocessing.sequence as ps

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth Enabled.")
    except RuntimeError as e:
        print(e)

tf.random.set_seed(42)
np.random.seed(42)


GPU Memory Growth Enabled.


## 2. Load Dataset
We read the dataset using Pandas and drop any missing values to maintain data integrity.


In [3]:
data = pd.read_csv('/kaggle/input/datasets/amanbanik/dataset-english-hindi/Dataset_English_Hindi.csv')
data.columns = ['English', 'Hindi']
data = data.dropna()

english_sentences = data['English'].astype(str).tolist()
hindi_sentences = data['Hindi'].astype(str).tolist()

print(f"Total sentences: {len(english_sentences)}")


Total sentences: 130162


## 3. Build and Save Vocabularies
We limit the vocabulary to the top 10,000 most common words. This prevents GPU memory OOM crashes and forces the model to generalize better. Rare words are replaced with `<UNK>`.


In [4]:
def build_vocab(sentences, max_vocab_size=10000):
    vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
    next_index = 4
    
    word_freq = collections.Counter()
    for sentence in sentences:
        word_freq.update(sentence.split())
        
    common_words = word_freq.most_common(max_vocab_size - len(vocab))
    
    for word, _ in common_words:
        vocab[word] = next_index
        next_index += 1
        
    return vocab

print("Building vocabularies...")
eng_vocab = build_vocab(english_sentences, max_vocab_size=10000)
hind_vocab = build_vocab(hindi_sentences, max_vocab_size=10000)

print(f"English vocab size: {len(eng_vocab)}")
print(f"Hindi vocab size: {len(hind_vocab)}")

with open('eng_vocab.json', 'w', encoding='utf-8') as f:
    json.dump(eng_vocab, f, ensure_ascii=False)
with open('hind_vocab.json', 'w', encoding='utf-8') as f:
    json.dump(hind_vocab, f, ensure_ascii=False)



Building vocabularies...
English vocab size: 10000
Hindi vocab size: 10000


## 4. Sequence Padding
Convert sentences to integer sequences. Unknown words use `<UNK>`. We cap length to 30.


In [5]:
def sentence_to_number(sentence, vocab):
    words = sentence.split()
    unk_id = vocab['<UNK>']
    numbers = [vocab.get(word, unk_id) for word in words]
    return [vocab['<SOS>']] + numbers + [vocab['<EOS>']]

def prepare_data(eng_sentences, hind_sentences, eng_vocab, hind_vocab, max_length=30):
    src_number = [sentence_to_number(sent, eng_vocab) for sent in eng_sentences]
    tgt_number = [sentence_to_number(sent, hind_vocab) for sent in hind_sentences]

    src_padded = ps.pad_sequences(src_number, padding='post', maxlen=max_length, truncating='post', value=eng_vocab['<PAD>'])
    tgt_padded = ps.pad_sequences(tgt_number, padding='post', maxlen=max_length, truncating='post', value=hind_vocab['<PAD>'])
    return src_padded, tgt_padded

print("Padding sequences...")
src_data, tgt_data = prepare_data(english_sentences, hindi_sentences, eng_vocab, hind_vocab)



Padding sequences...


## 5. Batching with tf.data.Dataset
Create a scalable dataset pipeline that handles shuffling and batching, preventing Out-Of-Memory (OOM) errors.


In [6]:
BATCH_SIZE = 32 
BUFFER_SIZE = 10000 

dataset = tf.data.Dataset.from_tensor_slices((src_data, tgt_data))
dataset = dataset.shuffle(BUFFER_SIZE, seed=42).batch(BATCH_SIZE, drop_remainder=True)



I0000 00:00:1783411422.068972      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783411422.072046      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


## 6. Model Architecture
Below we define the Encoder, Decoder, and the Seq2Seq wrapper in their respective classes.


### 6.1 The Encoder
Encodes the source sequence into a context vector (hidden and cell states).


In [7]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(Encoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embed_size)
        self.lstm = tf.keras.layers.LSTM(hidden_size, return_state=True)

    def call(self, x):
        embedded = self.embedding(x)
        _, hidden, cell = self.lstm(embedded)
        return hidden, cell



### 6.2 The Decoder
Decodes the context vector into the target sequence, one step at a time.


In [8]:
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(Decoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embed_size)
        self.lstm = tf.keras.layers.LSTM(hidden_size, return_state=True, return_sequences=True)
        self.dense = tf.keras.layers.Dense(vocab_size)

    def call(self, x, hidden, cell):
        embedded = self.embedding(x)
        lstm_output, hidden, cell = self.lstm(embedded, initial_state=[hidden, cell])
        output = self.dense(lstm_output)
        return output, hidden, cell



### 6.3 The Seq2Seq Wrapper
Combines the Encoder and Decoder, applying teacher forcing during training. Uses raw tf.while_loop to bypass Jupyter AutoGraph limitations.


In [9]:
class Seq2Seq(tf.keras.Model):
    def __init__(self, encoder, decoder, tgt_vocab_size):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.tgt_vocab_size = tgt_vocab_size

    def call(self, inputs, training=False):
        src, tgt = inputs[0], inputs[1]
        batch_size = tf.shape(src)[0]
        tgt_len = tf.shape(tgt)[1] 
        
        hidden, cell = self.encoder(src)
        
        # Raw tf.while_loop guarantees a dynamic loop graph without AutoGraph tracing issues
        i0 = tf.constant(1)
        input_token0 = tgt[:, 0:1] 
        all_outputs0 = tf.TensorArray(tf.float32, size=tgt_len - 1)

        def cond(i, input_token, h, c, all_outputs):
            return i < tgt_len
            
        def body(i, input_token, h, c, all_outputs):
            output, h, c = self.decoder(input_token, h, c)
            all_outputs = all_outputs.write(i - 1, output)
            
            if training:
                next_input = tf.expand_dims(tgt[:, i], 1)
            else:
                next_input = tf.argmax(output, axis=-1, output_type=tf.int32)
                
            return i + 1, next_input, h, c, all_outputs

        _, _, _, _, final_outputs = tf.while_loop(
            cond, body,
            loop_vars=[i0, input_token0, hidden, cell, all_outputs0]
        )

        outputs_stacked = final_outputs.stack()
        outputs_stacked = tf.squeeze(outputs_stacked, axis=2) 
        return tf.transpose(outputs_stacked, perm=[1, 0, 2]) 



## 7. Initialize Model
Set hyperparameters and initialize the model architecture.


In [10]:
embed_size = 256
hidden_size = 512
input_vocab_size = len(eng_vocab)
output_vocab_size = len(hind_vocab)

encoder = Encoder(input_vocab_size, embed_size, hidden_size)
decoder = Decoder(output_vocab_size, embed_size, hidden_size)
model = Seq2Seq(encoder, decoder, tgt_vocab_size=output_vocab_size)

dummy_src = tf.zeros((BATCH_SIZE, 30), dtype=tf.int32)
dummy_tgt = tf.zeros((BATCH_SIZE, 30), dtype=tf.int32)
_ = model([dummy_src, dummy_tgt], training=False)



## 8. Training Logic
Define the optimizer, loss function, and the custom training step using `tf.GradientTape`.


In [11]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

@tf.function
def train_step(src, tgt):
    with tf.GradientTape() as tape:
        predictions = model([src, tgt], training=True)
        target_labels = tgt[:, 1:] 
        seq_length = target_labels.shape[1]
        predictions = predictions[:, :seq_length, :]

        mask = tf.cast(target_labels != hind_vocab["<PAD>"], tf.float32)
        loss = loss_fn(target_labels, predictions)
        loss = loss * mask

        total_loss = tf.reduce_sum(loss)
        total_token = tf.reduce_sum(mask)
        mean_loss = total_loss / total_token

    gradients = tape.gradient(mean_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return mean_loss



## 9. Execute Training Loop with Early Stopping
Train the model up to 30 epochs, monitoring validation loss.


In [12]:
EPOCHS = 30
PATIENCE = 3
best_loss = float('inf')
patience_counter = 0

print('Starting Training...')

for epoch in range(EPOCHS):
    total_loss = 0.0
    steps = 0
    for batch_src, batch_tgt in dataset:
        loss = train_step(batch_src, batch_tgt)
        total_loss += float(loss.numpy())
        steps += 1
        
        if steps % 100 == 0:
            print(f"Epoch {epoch+1}, Step {steps}, Loss: {loss.numpy():.4f}")
            
    avg_loss = total_loss / steps
    print(f'\nEpoch: {epoch+1} | Average Loss: {avg_loss:.4f}')
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        model.save_weights('seq2seq_weights.weights.h5')
        print(f"[*] Loss improved to {best_loss:.4f}. Model weights saved.\n")
    else:
        patience_counter += 1
        print(f"[!] No improvement in loss. Patience: {patience_counter}/{PATIENCE}\n")
        if patience_counter >= PATIENCE:
            print("=> Early stopping triggered! Training stopped.")
            break



Starting Training...
Epoch 1, Step 100, Loss: 6.4202
Epoch 1, Step 200, Loss: 6.0008
Epoch 1, Step 300, Loss: 6.1939
Epoch 1, Step 400, Loss: 5.6171
Epoch 1, Step 500, Loss: 5.5919
Epoch 1, Step 600, Loss: 5.5001
Epoch 1, Step 700, Loss: 5.4981
Epoch 1, Step 800, Loss: 5.4516
Epoch 1, Step 900, Loss: 5.5089
Epoch 1, Step 1000, Loss: 5.5258
Epoch 1, Step 1100, Loss: 5.5094
Epoch 1, Step 1200, Loss: 5.2936
Epoch 1, Step 1300, Loss: 5.3111
Epoch 1, Step 1400, Loss: 5.1636
Epoch 1, Step 1500, Loss: 4.8987
Epoch 1, Step 1600, Loss: 5.2675
Epoch 1, Step 1700, Loss: 4.9445
Epoch 1, Step 1800, Loss: 4.8979
Epoch 1, Step 1900, Loss: 4.8759
Epoch 1, Step 2000, Loss: 5.0591
Epoch 1, Step 2100, Loss: 4.8521
Epoch 1, Step 2200, Loss: 5.1173
Epoch 1, Step 2300, Loss: 4.9806
Epoch 1, Step 2400, Loss: 5.0843
Epoch 1, Step 2500, Loss: 4.7445
Epoch 1, Step 2600, Loss: 4.8972
Epoch 1, Step 2700, Loss: 4.7219
Epoch 1, Step 2800, Loss: 4.9082
Epoch 1, Step 2900, Loss: 4.6551
Epoch 1, Step 3000, Loss: 4.697

KeyboardInterrupt: 

# ~ 2 hrs passed.
> not procedding more.